# Managing What the Agent Sees

An LLM's context window is its working memory — the complete set of tokens the model can see at generation time. Every prior message, every tool result, every line of the system prompt must fit within this window. For models like Claude, the limit is 200,000 tokens — the size of a small novel. That sounds large until you consider what fills it: a single `read_file` call on a non-trivial module might return 500 tokens; ten such calls in a debugging session produce 5,000 tokens of tool results alone. Add multi-turn conversation history, and the context fills faster than intuition suggests.

This creates two compounding problems. The first is cost: every token in the prompt is billed on each turn, so a growing context means an accelerating bill. The second is quality. Liu et al. (2023) demonstrated the "lost in the middle" phenomenon: models attend strongly to tokens near the beginning and end of the context, but performance degrades for information buried in the middle. A long, unmanaged conversation history is not just expensive — it actively hurts the model's ability to reason about the task at hand.

We study three strategies the CDA library uses to manage this. Token estimation gives us a fast, dependency-free proxy for context size. Mechanical pruning is the deterministic fallback: partition the conversation into system, middle, and tail; remove tool results first, then old assistant turns, until the estimated size drops below a threshold. LLM-powered compaction is the higher-fidelity approach: summarize the middle portion into a structured narrative that preserves key decisions, file paths, and unresolved issues. We also examine how the system prompt is assembled from modular sections — since it is a fixed overhead on every single turn.

## The Context Window Problem

To build intuition for when management becomes necessary, we create a `ContextManager` with a small context window and simulate a conversation growing from idle to critical.

**Setup.** Imports and a `Config` with a toy 4,096-token context window for demonstration:

In [ ]:
import os
import json
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

Importing from the CDA library:

In [ ]:
from notebooks.agent.config import Config, ModelConfig
from notebooks.agent.context import ContextManager
from notebooks.agent.compaction import ChatCompactor
from notebooks.agent.prompts import build_system_prompt
from notebooks.agent.client import LLMClient
from notebooks.agent.events import TokenUsage

We create a `ContextManager` with a small context window to make threshold crossings visible at token counts we can reason about directly:

In [ ]:
config = Config(model=ModelConfig(context_window=4096))
cm = ContextManager(config)

print(cm.get_context_stats())

Now we simulate context filling up by manually setting `token_count` and observing when the two thresholds fire:

In [ ]:
# Simulate context filling up
for n_tokens in [500, 1000, 2000, 3500, 3800]:
    cm.token_count = n_tokens
    stats = cm.get_context_stats()
    print(
        f"{n_tokens:5d} tokens | {stats['usage_percent']:5.1f}% "
        f"| compact={stats['needs_compaction']} "
        f"| prune={stats['needs_pruning']}"
    )

The compaction threshold fires at 80% ($0.8 \times 4096 = 3277$ tokens) and the hard-pruning threshold at 90% ($0.9 \times 4096 = 3686$ tokens). Both thresholds are configurable at `ContextManager` construction time.

## Token Estimation

Before we can decide whether to prune or compact, we need a count of how many tokens the current conversation consumes. The exact answer requires a model-specific tokenizer — `tiktoken` for GPT family models, a sentencepiece variant for Claude. Loading and running a tokenizer is an unnecessary dependency for what is essentially a threshold check.

The CDA library uses a simple heuristic instead:

$$\text{tokens}(m) = \lfloor |\text{content}| / 4 \rfloor + 10$$

where $|\text{content}|$ is the character length of the message content string, and the $+10$ accounts for role and formatting overhead. One token is approximately four characters in English text — a rough but durable rule of thumb. The per-message overhead of 10 tokens is a fixed budget for the `{"role": ..., "content": ...}` envelope.

We apply `estimate_message_tokens()` to a short conversation to see the heuristic in action:

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful coding assistant..."},
    {"role": "user", "content": "Help me refactor the authentication module"},
    {"role": "assistant", "content": "I'll start by reading the current implementation."},
    {"role": "tool", "content": "```python\n# 500 lines of code here...\n```" + "x" * 2000},
    {"role": "assistant", "content": "The auth module uses JWT. Here's what I'll change..."},
]

cm_fresh = ContextManager(Config())
total = sum(cm_fresh.estimate_message_tokens(m) for m in messages)
print(f"Estimated total: {total} tokens")
for i, m in enumerate(messages):
    est = cm_fresh.estimate_message_tokens(m)
    chars = len(str(m.get("content", "")))
    print(f"  msg[{i}] role={m['role']:10s}  chars={chars:5d}  →  est={est:4d} tokens")

The tool result (message 3) dominates — 2,000 padding characters plus the code fence contribute most of the estimated budget. This is characteristic of real agent sessions: a few verbose tool results dwarf the conversational turns.

:::{.callout-note}
The 4-characters-per-token heuristic is intentionally rough. For ASCII English prose the true ratio is around 3.8–4.2 characters per token; for code and structured data it can be lower (more tokens per character). The estimate is good enough for pruning decisions — we are trying to stay under a threshold, not bill for tokens — and it avoids a tokenizer dependency entirely.

:::

## Mechanical Pruning

When the estimated token count crosses the hard-pruning threshold (90% of the context window), the CDA library falls back to a deterministic strategy: partition the message list into **system** (index 0), **middle** (everything between system and the protected tail), and **tail** (the last `keep_last` messages), then selectively remove from the middle until the estimated total drops below the compaction threshold.

The removal order matters. Tool results are targeted first because they tend to be the largest messages — file contents, command output, JSON blobs — and because the assistant's subsequent response already summarizes the relevant information. Old assistant turns are removed next. The system message and the tail are never touched, preserving the model's grounding and the most recent context it needs for the next turn.

We build a synthetic conversation with 15 turns of tool use to observe the effect:

In [ ]:
def make_conversation(n_turns: int) -> list[dict]:
    msgs = [{"role": "system", "content": "You are a coding agent. " * 20}]
    for i in range(n_turns):
        msgs.append({"role": "user", "content": f"Task {i}: do something"})
        msgs.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": f"call_{i}",
                "type": "function",
                "function": {
                    "name": "read_file",
                    "arguments": f'{{"path": "file_{i}.py"}}'
                }
            }]
        })
        msgs.append({
            "role": "tool",
            "tool_call_id": f"call_{i}",
            "content": f"# file_{i}.py\n" + "x = 1\n" * 100
        })
        msgs.append({"role": "assistant", "content": f"Done with task {i}. The result is..."})
    return msgs

conversation = make_conversation(15)
print(f"Original: {len(conversation)} messages")
print(f"Estimated tokens: {sum(cm.estimate_message_tokens(m) for m in conversation)}")

Applying `prune_messages()` with a 4,096-token context window and a `keep_last=4` tail:

In [ ]:
cm_small = ContextManager(Config(model=ModelConfig(context_window=4096)), keep_last=4)
pruned = cm_small.prune_messages(conversation)
print(f"After pruning: {len(pruned)} messages")
print(f"Estimated tokens: {sum(cm_small.estimate_message_tokens(m) for m in pruned)}")
print(f"\nMessages kept:")
for m in pruned:
    content_preview = str(m.get("content", ""))[:60].replace("\n", " ")
    print(f"  [{m['role']:10s}]  {content_preview}...")

The pruner removes tool results from the middle first, then old assistant turns, until the estimate fits the budget. The tail (last 4 messages) is always preserved verbatim.

:::{.callout-caution}
Mechanical pruning is lossy. A tool result discarded from the middle might contain a critical error message — a stack trace, a failed test output, a filesystem path — that the model needs to reason about the current task. Pruning assumes the tail contains enough context to proceed. This assumption holds for well-structured tasks but can cause regressions in long debugging sessions where the root cause was established early in the conversation.

:::

## LLM-Powered Compaction

Compaction takes a different approach: instead of discarding the middle, it summarizes it. `ChatCompactor.compact()` keeps the system message (index 0) and the last `keep_last` messages verbatim, then replaces everything in between with a single summary message generated by a secondary LLM call.

The result is a three-part message list:

$$[\underbrace{\text{system\_msg}}_{\text{always kept}}, \underbrace{\text{summary\_msg}}_{\text{LLM-generated}}, \underbrace{\text{tail}_{1}, \ldots, \text{tail}_{k}}_{\text{keep\_last messages}}]$$

The summary message uses a `"user"` role and is wrapped in a structured marker:

```
[Context Summary]

<bullet-point summary of actions, files, decisions, unresolved issues>

[End of Summary — conversation continues below]
```

The summarizer is instructed to preserve file paths, error messages, short code snippets, and any pending tasks — the information a developer would write down before taking a break. It targets under 500 words and uses bullet points for scannability.

We run `compact()` on a realistic multi-turn conversation involving JWT refactoring and a failing test:

In [ ]:
config_live = Config()
client = LLMClient(config_live)
compactor = ChatCompactor(client, config_live)

convo = [
    {"role": "system", "content": "You are a coding agent."},
    {"role": "user", "content": "Refactor the authentication module to use JWT."},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [{"id": "c1", "type": "function", "function": {
            "name": "read_file", "arguments": '{"path": "auth.py"}'
        }}]
    },
    {
        "role": "tool",
        "tool_call_id": "c1",
        "content": "# auth.py\nclass AuthManager:\n    def login(self, user, password):\n        # TODO: implement\n        return None"
    },
    {"role": "assistant", "content": "The auth module is incomplete. I'll implement JWT-based login."},
    {"role": "user", "content": "Also add a logout endpoint."},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [{"id": "c2", "type": "function", "function": {
            "name": "write_file",
            "arguments": '{"path": "auth.py", "content": "import jwt\\n..."}'
        }}]
    },
    {
        "role": "tool",
        "tool_call_id": "c2",
        "content": "Wrote 342 bytes to auth.py"
    },
    {"role": "assistant", "content": "JWT login implemented. Now adding logout."},
    {"role": "user", "content": "Run the tests."},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [{"id": "c3", "type": "function", "function": {
            "name": "shell",
            "arguments": '{"command": "pytest tests/test_auth.py"}'
        }}]
    },
    {
        "role": "tool",
        "tool_call_id": "c3",
        "content": "PASSED tests/test_auth.py::test_login\nFAILED tests/test_auth.py::test_logout"
    },
    {"role": "assistant", "content": "Login test passes. Logout test fails — fixing now."},
]

print(f"Original: {len(convo)} messages")
compacted = await compactor.compact(convo, keep_last=4)
print(f"After compaction: {len(compacted)} messages")
print("\nThe summary message:")
print(compacted[1]["content"])

The compacted conversation has three messages: system, summary, and the last four turns. The summary preserves the key facts: what was attempted, what succeeded, what failed, and what is still pending. A fresh context built on this summary lets the model continue the debugging task without losing the thread.

Compare this to what mechanical pruning would produce on the same conversation: it would drop the tool results (removing the test output) and possibly the early assistant turns (removing the JWT decision), leaving the model with less context than it needs to diagnose the logout failure.

:::{.callout-note}
The CDA library uses a two-tier strategy: attempt compaction at 80% of the context window (the `compaction_threshold`), and fall back to hard pruning at 90% (the `pruning_threshold`). Compaction is preferred because it preserves information; pruning is the safety net for when compaction is unavailable or too slow.

:::

We can make the two-tier logic explicit by simulating token growth and checking which action would be triggered at each level:

In [ ]:
cm_demo = ContextManager(Config(model=ModelConfig(context_window=10_000)))
print(f"{'tokens':>8}  {'pct':>6}  action")
print("-" * 30)
for tokens in [0, 3_000, 6_000, 7_000, 8_500, 9_200]:
    cm_demo.token_count = tokens
    pct = tokens / 10_000 * 100
    needs_c = cm_demo.needs_compaction()
    needs_p = cm_demo.needs_pruning()
    if needs_p:
        action = "prune (fallback)"
    elif needs_c:
        action = "compact"
    else:
        action = "OK"
    print(f"{tokens:>8}  {pct:>5.0f}%  {action}")

At 8,500 tokens (85% of the window) compaction is triggered but pruning is not — the library will attempt a summary. At 9,200 tokens (92%) both flags are set — the hard pruning fallback activates.

## System Prompt Assembly

The system prompt is a fixed cost on every LLM call. For a 200,000-token context window, a 2,000-token system prompt is only 1% — but across a 100-turn conversation, those 2,000 tokens appear in every prompt, totaling 200,000 tokens of system prompt cost alone. Design matters.

`build_system_prompt()` assembles the prompt from seven modular sections joined with `"\n\n"`:

- **Identity:** Role description and capability summary. Always included.
- **Environment:** Date, OS, shell, and working directory. Grounds the model in runtime reality — important for shell commands and file paths.
- **Tool guidelines:** One-line description for each available tool, plus five best practices: read before editing, search before acting, surgical edits, shell for commands, parallelism. Included only if tools are provided.
- **Security:** Five guardrails — no exposed secrets, path validation, cautious commands, prompt injection defense, no untrusted execution. Always included.
- **Developer instructions:** Project-specific guidance injected by the developer (e.g., "use Python 3.13", "run ruff before committing"). Included only if non-null.
- **User instructions:** Per-session customization from the user. Included only if non-null.
- **Operational guidelines:** Tone, workflow steps (Understand → Plan → Implement → Verify), error recovery posture. Always included.

We generate prompts under two configs to see how the sections compose:

In [ ]:
from notebooks.agent.tools.registry import create_default_registry

registry = create_default_registry(Config())
tools = registry.get_tools()  # <1>

config_prompt = Config(
    developer_instructions="Use Python 3.13. Run ruff check before committing.",
    user_instructions="Prefer concise answers.",
)

prompt = build_system_prompt(config_prompt, tools)
print(f"System prompt length: {len(prompt)} characters")
print(f"Estimated tokens:     {ContextManager.estimate_message_tokens({'content': prompt})}")
print("\n" + "=" * 60)
print(prompt[:2000])
print("...")

1. `registry.get_tools()` returns the public list of `Tool` objects (respecting `config.allowed_tools` if set). We pass these to `build_system_prompt()` which formats each tool's name and description into the tool guidelines section.

We compare the full prompt against a minimal one — no tools, no custom instructions — to see how much each section contributes:

In [ ]:
config_minimal = Config()
prompt_minimal = build_system_prompt(config_minimal, [])

print(f"Minimal prompt: {len(prompt_minimal):5d} chars  ~{len(prompt_minimal)//4:4d} tokens")
print(f"Full prompt:    {len(prompt):5d} chars  ~{len(prompt)//4:4d} tokens")
print(f"Tool guidelines add: ~{(len(prompt) - len(prompt_minimal))//4} tokens")

:::{.callout-note}
System prompt token cost multiplies with every conversation turn. A 3,000-token system prompt on a 100-turn conversation accounts for 300,000 tokens of prompt cost — more than the entire 200,000-token context window. This is not a hypothetical: at standard OpenRouter pricing, an unoptimized system prompt in a long agent session can double the total bill. The modular design of `build_system_prompt()` makes it straightforward to strip sections (e.g., remove tool guidelines when the agent is in read-only mode) when token economy matters.

:::

---

■